In [ ]:
{
 "cells": [
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "setup",
   "metadata": {},
   "outputs": [],
   "source": [
    "!pip -q install rpy2\n",
    "%load_ext rpy2.ipython"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "packages",
   "metadata": {},
   "outputs": [],
   "source": [
    "%%R\n",
    "options(repos = c(CRAN = \"https://cloud.r-project.org\"))\n",
    "\n",
    "if (!require(\"pacman\")) install.packages(\"pacman\")\n",
    "pacman::p_load(tidyverse, ggplot2, dplyr, lubridate)"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "read-header",
   "metadata": {},
   "source": [
    "# Read and combine v1996 + v2010 data"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "read-combine",
   "metadata": {},
   "outputs": [],
   "source": "%%R\nfinal.hcris.v1996=read_csv('../data/output/HCRIS_Data_v1996.csv')\nfinal.hcris.v2010=read_csv('../data/output/HCRIS_Data_v2010.csv')\n\n## create missing variables for columns introduced in v2010 of hcris forms\nfinal.hcris.v1996 = final.hcris.v1996 %>%\n  mutate(hvbp_payment=NA, hrrp_payment=NA)\n\n## combine v1996 and v2010 hcris forms, and sort by provider_number/year\nfinal.hcris=rbind(final.hcris.v1996,final.hcris.v2010) %>%\n  mutate(fy_end=mdy(fy_end),fy_start=mdy(fy_start),\n         date_processed=mdy(date_processed),date_created=mdy(date_created),\n         tot_discounts=abs(tot_discounts), hrrp_payment=abs(hrrp_payment)) %>%\n  mutate(fyear=year(fy_end)) %>%\n  arrange(provider_number,fyear) %>%\n  select(-year)\n\n## count of hospitals/provider_number by year\nfinal.hcris %>% group_by(fyear) %>% count()"
  },
  {
   "cell_type": "markdown",
   "id": "clean-header",
   "metadata": {},
   "source": [
    "# Clean duplicate reports"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "clean",
   "metadata": {},
   "outputs": [],
   "source": "%%R\n## create count of reports by hospital fiscal year\nfinal.hcris =\n  final.hcris %>%\n  add_count(provider_number, fyear, name=\"total_reports\")\n\n## create running total of reports\nfinal.hcris =\n  final.hcris %>% \n  group_by(provider_number, fyear) %>%\n  mutate(report_number=row_number())\n\n## identify hospitals with only one report per fiscal year \n## this will be the first set of hospitals in the final dataset\nunique.hcris1 =\n  final.hcris %>%\n  filter(total_reports==1) %>%\n  select(-report, -total_reports, -report_number, -npi, -status) %>%\n  mutate(source='unique reports')\n\n\n## identify hospitals with multiple reports per fiscal year\nduplicate.hcris = \n  final.hcris %>%\n  filter(total_reports>1) %>%\n  mutate(time_diff=fy_end-fy_start)\n\n## calculate elapsed time between fy start and fy end for hospitals with multiple reports\nduplicate.hcris = \n  duplicate.hcris %>% \n  group_by(provider_number, fyear) %>%\n  mutate(total_days=sum(time_diff))\n\n## if the elapsed time within a fy sums to around 365, then just take the total of the two\n## this will be the second set of hospitals in the final dataset\nunique.hcris2 = \n  duplicate.hcris %>%\n  filter(total_days<370) %>%\n  group_by(provider_number, fyear) %>%\n  mutate(hrrp_payment=if_else(is.na(hrrp_payment),0,hrrp_payment),\n         hvbp_payment=if_else(is.na(hvbp_payment),0,hvbp_payment)) %>%\n  summarize(beds=max(beds), tot_charges=sum(tot_charges), tot_discounts=sum(tot_discounts),\n            tot_operating_exp=sum(tot_operating_exp), ip_charges=sum(ip_charges),\n            icu_charges=sum(icu_charges), ancillary_charges=sum(ancillary_charges),\n            tot_discharges=sum(tot_discharges), mcare_discharges=sum(mcare_discharges),\n            mcaid_discharges=sum(mcaid_discharges), tot_mcare_payment=sum(tot_mcare_payment),\n            secondary_mcare_payment=sum(secondary_mcare_payment), hvbp_payment=sum(hvbp_payment),\n            hrrp_payment=sum(hrrp_payment), fy_start=min(fy_start), fy_end=max(fy_end),\n            date_processed=max(date_processed), date_created=min(date_created), \n            street=first(street), city=first(city), state=first(state),\n            zip=first(zip), county=first(county)) %>%\n  mutate(source='total for year')\n\n## identify hospitals with more than one report and with elapsed time exceeding 370 days\nduplicate.hcris2 =\n  duplicate.hcris %>%\n  filter(total_days>=370) %>%\n  mutate(max_days=max(time_diff), max_date=max(fy_end))\n\n## identify hospitals with one report (out of multiple) that appears to cover the full year\n## this will be the third set of hospitals in the final dataset\nunique.hcris3 = \n  duplicate.hcris2 %>%\n  filter(max_days==time_diff, time_diff>360, max_date==fy_end) %>%\n  select(-report, -total_reports, -report_number, -npi, -status, -max_days, -time_diff, -total_days, -max_date) %>%\n  mutate(source='primary report')\n\n## identify remaining hospitals\nduplicate.hcris3=anti_join(duplicate.hcris2, unique.hcris3, by=c(\"provider_number\", \"fyear\"))\nduplicate.hcris3 =\n  duplicate.hcris3 %>%\n  mutate(total_days=as.integer(total_days), time_diff=as.integer(time_diff)) %>%\n  mutate_at(c(\"tot_charges\",\"tot_discounts\", \"tot_operating_exp\", \"ip_charges\",\n              \"icu_charges\", \"ancillary_charges\", \"tot_discharges\", \"mcare_discharges\",\n              \"mcaid_discharges\", \"tot_mcare_payment\", \"secondary_mcare_payment\",\n              \"hvbp_payment\", \"hrrp_payment\"),list(~ .*(time_diff/total_days)))\n\n## form weighted average of values for each fiscal year\nunique.hcris4 = \n  duplicate.hcris3 %>%\n  group_by(provider_number, fyear) %>%\n  mutate(hrrp_payment=if_else(is.na(hrrp_payment),0,hrrp_payment),\n         hvbp_payment=if_else(is.na(hvbp_payment),0,hvbp_payment)) %>%\n  summarize(beds=max(beds), tot_charges=sum(tot_charges), tot_discounts=sum(tot_discounts),\n            tot_operating_exp=sum(tot_operating_exp), ip_charges=sum(ip_charges),\n            icu_charges=sum(icu_charges), ancillary_charges=sum(ancillary_charges),\n            tot_discharges=sum(tot_discharges), mcare_discharges=sum(mcare_discharges),\n            mcaid_discharges=sum(mcaid_discharges), tot_mcare_payment=sum(tot_mcare_payment),\n            secondary_mcare_payment=sum(secondary_mcare_payment), hvbp_payment=sum(hvbp_payment),\n            hrrp_payment=sum(hrrp_payment), fy_start=min(fy_start), fy_end=max(fy_end),\n            date_processed=max(date_processed), date_created=min(date_created), \n            street=first(street), city=first(city), state=first(state),\n            zip=first(zip), county=first(county)) %>%\n  mutate(source='weighted_average')"
  },
  {
   "cell_type": "markdown",
   "id": "save-header",
   "metadata": {},
   "source": [
    "# Save final per-year CSVs"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "save",
   "metadata": {},
   "outputs": [],
   "source": [
    "%%R\n",
    "final.hcris.data=rbind(unique.hcris1, unique.hcris2, unique.hcris3, unique.hcris4)\n",
    "final.hcris.data =\n",
    "  final.hcris.data %>%\n",
    "  rename(year=fyear) %>%\n",
    "  arrange(provider_number, year)\n",
    "\n",
    "for (y in sort(unique(final.hcris.data$year))) {\n",
    "  final.hcris.data %>%\n",
    "    filter(year==y) %>%\n",
    "    write_csv(paste0('../data/output/data-', y, '.csv'))\n",
    "}\n",
    "\n",
    "cat('Years written:', sort(unique(final.hcris.data$year)), '\\n')\n",
    "cat('Total rows:', nrow(final.hcris.data), '\\n')"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3 (ipykernel)",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.12.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}